In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupShuffleSplit
scaler = MinMaxScaler()
df_courses_tasks = pd.read_csv("train/courses_tasks_train.csv")
df_activity_log = pd.read_csv("train/activity_log_train.csv")
df_students = pd.read_csv("train/students_train.csv")
df_task_marks = pd.read_csv("train/task_marks_train.csv")
df_courses = pd.read_csv("train/courses_train.csv")
df_labels = pd.read_csv("train/final_marks_train.csv")

In [2]:
df_labels['final_mark'] = (np.round(df_labels['final_mark'] / 10)).astype(int) # Adjust final marks.

student_ids_in_labels = df_labels['student_id'].unique() # Filter students that are not marked at all. 
filtered_students = df_students[df_students['student_id'].isin(student_ids_in_labels)]
filtered_students = filtered_students.drop_duplicates(subset='student_id', keep='first') # There were duplicated, keep the first, drop the second

In [3]:
# Students Pre processing
def students_preprocess(df):
    # DOB to age. forgot to normalize it tho
    df["dob"] = pd.to_datetime(df["dob"])
    today = pd.Timestamp(datetime.today().date())
    df["age"] = df["dob"].apply(lambda x: today.year - x.year - ((today.month, today.day) < (x.month, x.day)))
    df["age"] = scaler.fit_transform(df[["age"]])
    df.drop(columns=["dob"], inplace=True)

    # hot encoding of nationality and gender
    encoder = OneHotEncoder(sparse_output=False)  
    encoded = encoder.fit_transform(df[["gender", "nationality"]])

    encoded_cols = encoder.get_feature_names_out(["gender", "nationality"])
    df_encoded = pd.DataFrame(encoded, columns=encoded_cols, index=df.index)

    # merging
    df_final = pd.concat([df.drop(columns=["gender", "nationality"]), df_encoded], axis=1)
    return df_final
df_pp_students = students_preprocess(filtered_students)

In [4]:
# NEW - Breadth and intensity of study/engagement a student has with non graded tasks.
def course_task_non_grade_revised(df_courses_tasks, df_activity_log):
    """
    Computes student-level features based on engagement with non-graded tasks,
    but aggregated per course to maintain context.
    - non_graded_interactions_per_course: intensity of study
    - unique_non_graded_tasks_viewed_per_course: breadth of study
    
    Returns a dataframe with features for each student-course pair.
    """

    non_graded_tasks = df_courses_tasks[ # all non graded tasks - extra workload
        (df_courses_tasks["is_resource"] == True) | (df_courses_tasks["weight"] == 0)
    ]
    
    logs_non_graded = df_activity_log[ # filtering logs to see the interactions with these tasks
        df_activity_log["task_id"].isin(non_graded_tasks["task_id"])
    ].copy() # cop removes the warning
    
    if logs_non_graded.empty:
        # if no interaction, then return empty df accordingly
        return pd.DataFrame(columns=[
            "student_id",
            "course_id",
            "non_graded_interactions_per_course",
            "unique_non_graded_tasks_viewed_per_course"
        ])
    
    # Calculate the total number of interactions per student-course.
    activity_counts_per_course = ( # keep it consistent with the labels df, student_id, course_id merging
        logs_non_graded.groupby(["student_id", "course_id"])["task_id"]
        .count()
        .reset_index(name="non_graded_interactions_per_course")
    )
    
    task_coverage_per_course = ( # count the number of unique tasks vewed per student
        logs_non_graded.groupby(["student_id", "course_id"])["task_id"]
        .nunique()
        .reset_index(name="unique_non_graded_tasks_viewed_per_course")
    )

    features_per_course = activity_counts_per_course.merge( # merging the two new features
        task_coverage_per_course, on=["student_id", "course_id"], how="outer"
    )

    features_per_course = features_per_course.fillna(0) # if its nan fill it with 0 since they didnt interact anyway

    # will normalize later on
    return features_per_course
student_course_non_graded_features = course_task_non_grade_revised(df_courses_tasks, df_activity_log)

In [5]:
def get_interaction_features(task_marks, activity_log, courses_tasks):
    # Performance based
    performance_features = task_marks.groupby(['student_id', 'course_id']).agg( # performance based features
        avg_mark=('mark', 'mean'), # avg of the marks
        std_mark=('mark', 'std'), # std of the marks, removing one or the other affected the results so we kept both of them
        tasks_submitted=('task_id', 'count') # specific task ids and submittions
    ).reset_index()
    performance_features['std_mark'] = performance_features['std_mark'].fillna(0) # filling the nans(no submissions), with std, obviously it is better than avg so

    # Engagement based
    # getting engagement here, it differs from view and submit since some of the tasks are not graded but still gives some engagement and workload to student
    action_counts = activity_log.groupby(['student_id', 'course_id', 'action']).size().unstack(fill_value=0)
    action_counts = action_counts.rename(columns={'view': 'view_count', 'submit': 'submit_count'})

    # to make it easier on the model and get a better understanding of the data we ratio it 
    action_counts['total_activity'] = action_counts['view_count'] + action_counts['submit_count']
    # np.divide for faster and safer division
    action_counts['view_submit_ratio'] = np.divide(action_counts['view_count'], action_counts['submit_count'])
    action_counts['view_submit_ratio'] = action_counts['view_submit_ratio'].replace([np.inf, -np.inf], 0).fillna(0)
    engagement_features = action_counts.reset_index()
    
    # Timeliness based
    # stding the time format just in case
    activity_log['timestamp'] = pd.to_datetime(activity_log['timestamp'])
    courses_tasks['deadline'] = pd.to_datetime(courses_tasks['deadline'])

    submissions = activity_log[activity_log['action'] == 'submit'].copy() # only the submissions which is graded

    submission_deadlines = pd.merge( # merging submission with task deadlines
        submissions,
        courses_tasks[['task_id', 'deadline']],
        on='task_id',
        how='left'
    )
    #
    # calculate lateness in hours. positive means late, negative means early.
    submission_deadlines['lateness_hours'] = (submission_deadlines['timestamp'] - submission_deadlines['deadline']).dt.total_seconds() / 3600
    submission_deadlines['is_late'] = submission_deadlines['lateness_hours'] > 0
    
    timeliness_features = submission_deadlines.groupby(['student_id', 'course_id']).agg( #aggregating timeliness for each student x course
        avg_lateness_hours=('lateness_hours', 'mean'),
        max_lateness_hours=('lateness_hours', 'max'),
        num_late_submissions=('is_late', 'sum'),
        num_early_submissions=('is_late', lambda x: (1 - x).sum()) # early submissions count by inverting the is_late
    ).reset_index()
    # If a student was never late, max_lateness might be negative. Let's cap it at 0.
    timeliness_features['max_lateness_hours'] = timeliness_features['max_lateness_hours'].clip(lower=0) # if a student is never late then it gets negative which is no bueno, so cap is 0
    


    # Merging everything we got
    df_merged = pd.merge(performance_features, engagement_features, on=['student_id', 'course_id'], how='left')
    df_merged = pd.merge(df_merged, timeliness_features, on=['student_id', 'course_id'], how='left')

    # Fill any remaining NaNs with 0 (e.g., if a student has no submissions for timeliness)
    df_merged = df_merged.fillna(0)

    return df_merged
interaction_df = get_interaction_features(df_task_marks, df_activity_log,  df_courses_tasks)

In [6]:
def get_course_features(courses, courses_tasks):
    
    # Task and resource counts 
    # split the tasks into resource or assignment from is_resource
    task_counts = courses_tasks[courses_tasks['is_resource'] == False].groupby('course_id').size().rename('task_count')
    resource_counts = courses_tasks[courses_tasks['is_resource'] == True].groupby('course_id').size().rename('resource_count')

    # how hard/intense the course is
    course_features = courses[['course_id', 'ects', 'duration_months']].copy()

    # there was no courses with 0 duration but still just in case of test and hidden data later on.
    duration = course_features['duration_months'].replace(0, 1)
    course_features['course_intensity'] = course_features['ects'] / duration # straight up ects/duration of the course, this is how it is calculated IRL as well so, seems like a good fit
    
    # merging everything    
    course_features = course_features.merge(task_counts, on='course_id', how='left') # all is left to lose nothing
    course_features = course_features.merge(resource_counts, on='course_id', how='left')

    course_features = course_features.fillna(0) # if by chance there are no tasks for a course, dont wanna keep NaN during training
    
    # make srue everything is integer
    course_features['task_count'] = course_features['task_count'].astype(int)
    course_features['resource_count'] = course_features['resource_count'].astype(int)

    # Normalization lter like before
    return course_features


course_features_df = get_course_features(df_courses, df_courses_tasks)

In [7]:
# combine all the features accordingly student_id x course_id like in the final_marks
# again all is left to lose nothing
merged = interaction_df.merge(
    df_labels, 
    on=['student_id', 'course_id'], 
    how='left'  # or 'left' if you want to keep all course records
)
df_train = merged.merge(
    df_pp_students,
    on='student_id',
    how='left'  # or 'left' to keep all records
)
df_train = df_train.merge(
    course_features_df,
    on='course_id',
    how='left'  # or 'left' to keep all records
)
df_train = df_train.merge(
    student_course_non_graded_features,
    on=['student_id', 'course_id'],
    how='left'
)
df_train

,student_id,course_id,avg_mark,std_mark,tasks_submitted,submit_count,view_count,total_activity,view_submit_ratio,avg_lateness_hours,...,nationality_Spain,nationality_Sweden,nationality_USA,ects,duration_months,course_intensity,task_count,resource_count,non_graded_interactions_per_course,unique_non_graded_tasks_viewed_per_course
0,STU001C1A,CRS873520,73.0,5.656854,2,2,4,6,2.00,48.00,...,0.0,0.0,0.0,10,1,10.00,2,2,2,2
1,STU004FF9,CRS283DC7,0.0,0.000000,1,0,4,4,0.00,0.00,...,0.0,1.0,0.0,10,4,2.50,1,4,4,4
2,STU004FF9,CRS7BD731,85.0,0.000000,1,1,7,8,7.00,-120.00,...,0.0,1.0,0.0,9,4,2.25,1,6,6,6
3,STU00641A,CRSCCEF31,36.0,0.000000,1,1,4,5,4.00,336.00,...,0.0,0.0,0.0,3,4,0.75,1,3,3,3
4,STU0065A7,CRS675703,49.0,0.000000,1,0,20,20,0.00,0.00,...,0.0,0.0,0.0,6,4,1.50,1,4,20,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2955,STUFFDB9D,CRS94D1D4,13.5,27.000000,4,0,3,3,0.00,0.00,...,0.0,0.0,0.0,9,4,2.25,4,3,3,3
2956,STUFFE655,CRS7A2125,61.0,0.000000,1,0,14,14,0.00,0.00,...,0.0,0.0,0.0,10,4,2.50,1,3,14,3
2957,STUFFF26A,CRS9A7763,81.0,0.000000,1,1,6,7,6.00,120.00,...,0.0,0.0,0.0,6,4,1.50,1,5,5,5
2958,STUFFF768,CRS6F5FBE,32.0,0.000000,1,1,9,10,9.00,192.00,...,0.0,0.0,0.0,9,4,2.25,1,2,8,2


In [8]:
# Data prep
X = df_train.drop(columns=["student_id", "course_id", "final_mark"])
y = df_train["final_mark"]
groups = df_train["student_id"]  # Keep student_id for grouping

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=69)
train_idx, val_idx = next(gss.split(X, y, groups=groups))

In [9]:
# sanity check - you can ignoer this
X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]
y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

# Verify no student overlap
train_students = set(df_train.iloc[train_idx]["student_id"])
val_students = set(df_train.iloc[val_idx]["student_id"])
print(f"Students in train: {len(train_students)}")
print(f"Students in val: {len(val_students)}")
print(f"Overlap: {len(train_students & val_students)}") 

Students in train: 2192
Students in val: 548
Overlap: 0


In [10]:
def optimize_model_cls(trial, model_name, X_train, y_train, X_val, y_val):
    
    if model_name == "RandomForest":
        n_estimators = trial.suggest_int("n_estimators", 50, 800)
        max_depth = trial.suggest_int("max_depth", 3, 70)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 70)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 70)
        max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        bootstrap = trial.suggest_categorical("bootstrap", [True, False])
        
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=69,
            n_jobs=-1,
            warm_start=warm_start,
            bootstrap=bootstrap,
        )
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    preds = model.predict(X_val)
    
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average="weighted", zero_division=0) # zero_division=0 to avoid annoying warnings
    recall = recall_score(y_val, preds, average="weighted", zero_division=0)
    precision = precision_score(y_val, preds, average="weighted", zero_division=0)
    
    # metrics for the model
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("precision", precision)
    
    # minimizing F1
    return -f1

In [11]:
# Main optimization loop
models_to_optimize = ["RandomForest"]
best_models = {}

for model_name in models_to_optimize:
    print(f"\n{'='*50}")
    print(f"Optimizing {model_name}...")
    print(f"{'='*50}")
    
    study = optuna.create_study(direction="minimize")
    study.optimize(
        lambda trial: optimize_model_cls(trial, model_name, X_train, y_train, X_val, y_val),
        n_trials=100, 
        show_progress_bar=True
    )
    
    best_models[model_name] = {
        "best_params": study.best_params,
        "best_f1": study.best_trial.user_attrs.get("f1"),
        "best_accuracy": study.best_trial.user_attrs.get("accuracy"),
        "best_recall": study.best_trial.user_attrs.get("recall"),
        "best_precision": study.best_trial.user_attrs.get("precision")
    }
    
    print(f"\nBest F1-score for {model_name}: {best_models[model_name]['best_f1']:.4f}")
    print(f"Accuracy: {best_models[model_name]['best_accuracy']:.4f}")
    print(f"Recall: {best_models[model_name]['best_recall']:.4f}")
    print(f"Precision: {best_models[model_name]['best_precision']:.4f}")
    print(f"Best params: {study.best_params}")


[I 2025-10-03 19:13:31,446] A new study created in memory with name: no-name-1dae231b-ef0e-49a5-8f8c-b05c4621e242



Optimizing RandomForest...


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-10-03 19:13:31,617] Trial 0 finished with value: -0.8905004981414661 and parameters: {'n_estimators': 113, 'max_depth': 57, 'min_samples_split': 15, 'min_samples_leaf': 25, 'max_features': None, 'warm_start': True, 'bootstrap': True}. Best is trial 0 with value: -0.8905004981414661.
[I 2025-10-03 19:13:32,374] Trial 1 finished with value: -0.8799565975812069 and parameters: {'n_estimators': 596, 'max_depth': 57, 'min_samples_split': 40, 'min_samples_leaf': 2, 'max_features': None, 'warm_start': False, 'bootstrap': True}. Best is trial 0 with value: -0.8905004981414661.
[I 2025-10-03 19:13:32,494] Trial 2 finished with value: -0.6669605470958883 and parameters: {'n_estimators': 111, 'max_depth': 22, 'min_samples_split': 37, 'min_samples_leaf': 25, 'max_features': 'log2', 'warm_start': False, 'bootstrap': False}. Best is trial 0 with value: -0.8905004981414661.
[I 2025-10-03 19:13:33,058] Trial 3 finished with value: -0.46471661504344713 and parameters: {'n_estimators': 513, 'max

In [12]:
import joblib
best_model_name = max(best_models, key=lambda x: best_models[x]["best_f1"])
best_params = best_models[best_model_name]["best_params"]

print(f"Training final model: {best_model_name}")
print(f"With parameters: {best_params}")

if best_model_name == "RandomForest":
    final_model = RandomForestClassifier(
        **best_params,
        random_state=69,
        n_jobs=-1
    )

final_model.fit(X_train, y_train)
# Save the trained model
joblib.dump(final_model, 'best_model.pkl')
print("Model saved as 'best_model.pkl'")

Training final model: RandomForest
With parameters: {'n_estimators': 135, 'max_depth': 24, 'min_samples_split': 11, 'min_samples_leaf': 38, 'max_features': None, 'warm_start': True, 'bootstrap': True}
Model saved as 'best_model.pkl'
